# 🚀 Chapter 07-03b: PROJECT — Autonomous Research Agent

---

## 📖 The Mission

Build a research agent that can:
1. **Plan** what to research
2. **Search** for information
3. **Analyze** findings
4. **Write** a structured report
5. **Review** and refine if needed

This is a multi-step LangGraph workflow with cycles!

---

## 🧪 Step 1: Setup

In [ ]:
# !pip install langgraph langchain-openai langchain-core python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
print("✅ Setup complete!")

## 🧪 Step 2: Define Research State

In [ ]:
class ResearchState(TypedDict):
    messages: Annotated[list, add_messages]
    topic: str
    research_plan: str
    findings: str
    draft_report: str
    review_feedback: str
    revision_count: int
    status: str  # planning, researching, writing, reviewing, done

print("✅ ResearchState defined with 8 fields!")

## 🧪 Step 3: Define Agent Nodes

In [ ]:
def planner(state: ResearchState) -> dict:
    """Create a research plan based on the topic."""
    response = llm.invoke([
        SystemMessage(content="You are a research planner. Given a topic, create a concise 3-point research plan. Each point should be one line."),
        HumanMessage(content=f"Create a research plan for: {state['topic']}")
    ])
    print(f"  📋 Planner created plan")
    return {
        "research_plan": response.content,
        "status": "researching",
        "messages": [AIMessage(content=f"Plan: {response.content}")]
    }

def researcher(state: ResearchState) -> dict:
    """Research the topic based on the plan."""
    response = llm.invoke([
        SystemMessage(content="You are a researcher. Based on the plan, provide key findings with facts, data, and examples. Be specific and detailed."),
        HumanMessage(content=f"Topic: {state['topic']}\nPlan: {state['research_plan']}\nResearch this and provide findings.")
    ])
    print(f"  🔍 Researcher found information")
    return {
        "findings": response.content,
        "status": "writing",
        "messages": [AIMessage(content=f"Findings: {response.content[:100]}...")]
    }

def writer(state: ResearchState) -> dict:
    """Write a structured report from findings."""
    feedback_section = ""
    if state.get("review_feedback"):
        feedback_section = f"\nPrevious feedback to address: {state['review_feedback']}"
    
    response = llm.invoke([
        SystemMessage(content="You are a technical writer. Write a clear, structured report with Introduction, Key Findings, and Conclusion sections."),
        HumanMessage(content=f"Topic: {state['topic']}\nFindings: {state['findings']}{feedback_section}\nWrite a structured report.")
    ])
    print(f"  ✍️ Writer drafted report")
    return {
        "draft_report": response.content,
        "status": "reviewing",
        "messages": [AIMessage(content="Draft report written.")]
    }

def reviewer(state: ResearchState) -> dict:
    """Review the report and provide feedback."""
    response = llm.invoke([
        SystemMessage(content="""You are a strict editor. Review this report.
        If it's good enough, respond with exactly: APPROVED
        If it needs improvement, provide specific feedback (max 2 points)."""),
        HumanMessage(content=f"Review this report:\n{state['draft_report']}")
    ])
    approved = "APPROVED" in response.content.upper()
    print(f"  {'✅' if approved else '🔄'} Reviewer: {'Approved!' if approved else 'Needs revision'}")
    return {
        "review_feedback": response.content,
        "revision_count": state.get("revision_count", 0) + 1,
        "status": "done" if approved else "writing",
        "messages": [AIMessage(content=f"Review: {'Approved' if approved else 'Revision needed'}")]
    }

print("✅ All 4 agent nodes defined!")

## 🧪 Step 4: Define Routing Logic

In [ ]:
def review_router(state: ResearchState) -> Literal["writer", "end"]:
    """Route after review: revise or finish."""
    if state.get("status") == "done":
        return "end"
    if state.get("revision_count", 0) >= 3:  # Max 3 revisions
        print("  ⚠️ Max revisions reached, finishing.")
        return "end"
    return "writer"  # Send back for revision (CYCLE!)

print("✅ Router defined — max 3 revision cycles!")

## 🧪 Step 5: Build the Research Graph

In [ ]:
graph = StateGraph(ResearchState)

# Add nodes
graph.add_node("planner", planner)
graph.add_node("researcher", researcher)
graph.add_node("writer", writer)
graph.add_node("reviewer", reviewer)

# Linear flow: plan → research → write
graph.set_entry_point("planner")
graph.add_edge("planner", "researcher")
graph.add_edge("researcher", "writer")
graph.add_edge("writer", "reviewer")

# CYCLE: reviewer → writer (if needs revision) or → END
graph.add_conditional_edges("reviewer", review_router, {
    "writer": "writer",
    "end": END
})

# Compile
memory = MemorySaver()
research_app = graph.compile(checkpointer=memory)

print("✅ Research Agent compiled!")
try:
    print(research_app.get_graph().draw_ascii())
except Exception:
    print("Graph: planner → researcher → writer ↔ reviewer → END")

## 🧪 Step 6: Run the Research Agent!

In [ ]:
# Run with streaming to see each step
config = {"configurable": {"thread_id": "research-1"}}

print("🚀 Starting Research Agent...")
print("="*60)

result = research_app.invoke(
    {
        "messages": [HumanMessage(content="Research this topic")],
        "topic": "The impact of LLMs on software development in 2024",
        "research_plan": "",
        "findings": "",
        "draft_report": "",
        "review_feedback": "",
        "revision_count": 0,
        "status": "planning"
    },
    config
)

print("\n" + "="*60)
print(f"📊 Revisions: {result['revision_count']}")
print(f"📝 Status: {result['status']}")

In [ ]:
# Print the final report
print("📄 FINAL RESEARCH REPORT")
print("=" * 60)
print(result["draft_report"])

## ✍️ Practice Challenge

Extend the agent with a **"fact_checker"** node between researcher and writer that validates the findings. If facts seem wrong, route back to researcher.

<details>
<summary>💡 Hint</summary>

```python
def fact_checker(state):
    response = llm.invoke([...])
    if "VERIFIED" in response.content.upper():
        return {"status": "writing"}
    return {"status": "researching", "findings": ""}

# Add conditional edge from fact_checker
graph.add_conditional_edges("fact_checker", 
    lambda s: "writer" if s["status"]=="writing" else "researcher")
```

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Multi-step workflows: planner → researcher → writer → reviewer.│
├─────────────────────────────────────────────────────────────────────┤
│  📌 Revision cycle: reviewer → writer loop with max retries.       │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Rich state: TypedDict tracks progress across all nodes.        │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Conditional routing enables dynamic graph behavior.            │
├─────────────────────────────────────────────────────────────────────┤
│  📌 This pattern = foundation for production agent systems.        │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Chapter 08 — Multi-Agent Systems!

---

### 🎉 You Built a Research Agent!
Plan → Research → Write → Review → Revise — autonomous AI workflows. 🚀